In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# census_mx (worked-example helpers) lives in scripts/
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import pandas as pd
import numpy as np
import yaml

pd.set_option("display.max_columns", None)

# Local census parquet dir; set to None to fetch from the mxcensus mirror
DATA = None


We load the survey data, two tables, one for people and one for households.
We would like to use data from both tables simulataneously for imputation.

In [ ]:
from mxcensus.extended_personas import load_extended_personas
from mxcensus.extended_viviendas import load_extended_viviendas

# Columns to ignore for imputation, some are aggregated from other table
col_ignore_viv = ["DUE1_NUM", "DUE2_NUM", "ENT", "JEFE_EDAD"]
col_ignore_per = [
    "ENT",
    "IDENT_MADRE",
    "IDENT_PADRE",
    "IDENT_PAREJA",
    "IDENT_HIJO",
    "EDAD_MORIR_D",
    "EDAD_MORIR_M",
    "EDAD_MORIR_A",
    "EDAD_MORIR_TD",
    "FECHA_NAC_M",
    "QDIALECT_INALI",
    "NUMPER",
    "CAU_VER",
    "CAU_OIR",
    "CAU_CAMINAR",
    "CAU_RECORDAR",
    "CAU_BANARSE",
    "CAU_HABLAR",
    "CAU_MENTAL",
]

# Columns to transform into numerical (integer)
# After replacing "No especificado" with NaN
num_cols_viv = ["CUADORM", "TOTCUART"]
num_cols_per = [
    "FECHA_NAC_A",
    "ESCOACUM",
    "HIJOS_NAC_VIVOS",
    "HIJOS_FALLECIDOS",
    "HIJOS_SOBREVIV",
]

# Columns that are actually categorical but are coded as integer
cat_cols_viv = ["MUN", "LOC50K", "UPM"]
cat_cols_per = ["MUN", "LOC50K", "UPM", "ENT_PAIS_NAC"]

# ── Data loading ──────────────────────────────────────────────────────────────

print("Loading data...")
df_viv = (
    load_extended_viviendas(DATA / "Viviendas14.parquet")
    # .drop(columns=col_ignore_viv)
    # .replace("No especificado", np.nan)
)
df_per = (
    load_extended_personas(DATA / "Personas14.parquet")
    # .drop(columns=col_ignore_per)
    # .replace("No especificado", np.nan)
)


Loading data...


In [ ]:
df_viv.INGTRHOG_CAT.value_counts().sort_index() / len(df_viv) * 100

INGTRHOG_CAT
No recibe ingresos     3.115154
1-999                  0.904563
1,000-4,999            9.915087
5,000-9,999           28.711344
10,000-19,999         26.907760
20,000-39,999         10.864925
40,000-79,999          1.991610
80,000-149,999         0.327084
150,000yMas            0.086853
No especificado        0.226372
Blanco por pase       16.949247
Name: count, dtype: float64

In [ ]:
# Sin ingresos
# Revisar las no especificado, deberían ser no especificado
idx_no = df_viv.query("INGTRHOG_CAT == 'No recibe ingresos'").index
print(len(idx_no))
df_per_no = df_per.loc[idx_no][["EDAD", "CONACT_CAT", "INGTRMEN"]]
df_per_no.CONACT_CAT.value_counts()

6743


CONACT_CAT
No trabaja         8980
Trabaja            8930
Blanco por pase    4132
Buscó trabajo       139
No especificado      75
Name: count, dtype: int64

In [ ]:
# All households have al least one "Trabaja", all with 0 income.
# But there 75 "No especificado"
df_per_no.groupby("ID_VIV").filter(
    lambda df: np.any(df.CONACT_CAT.isin(["Trabaja"]))
).groupby("ID_VIV").INGTRMEN.sum()

ID_VIV
140010000034    0
140010000407    0
140010000538    0
140010000543    0
140010000556    0
               ..
141250000273    0
141250000359    0
141250000907    0
141250001266    0
141250001399    0
Name: INGTRMEN, Length: 6743, dtype: Int64

In [ ]:
df_per.loc[
    df_per_no.query("CONACT_CAT == 'No especificado'")
    .index.get_level_values(0)
    .unique()
][["EDAD", "CONACT_CAT", "INGTRMEN", "EDUC"]]

EDAD       CONACT_CAT  INGTRMEN  \
ID_VIV       ID_PERSONA                                           
140020000295 14002000029500001    38       No trabaja      <NA>   
             14002000029500002    42  No especificado      <NA>   
             14002000029500003    65       No trabaja      <NA>   
             14002000029500004    65          Trabaja         0   
             14002000029500005    12       No trabaja      <NA>   
...                              ...              ...       ...   
141210001789 14121000178900002    13       No trabaja      <NA>   
             14121000178900003    33    Buscó trabajo      <NA>   
             14121000178900004    50          Trabaja         0   
             14121000178900005     6  Blanco por pase      <NA>   
             14121000178900006    25  No especificado      <NA>   

                                            EDUC  
ID_VIV       ID_PERSONA                           
140020000295 14002000029500001    Secundaria_com  
             14002000029500002    Secundaria_com  
             14002000029500003      Primaria_com  
             14002000029500004     Sin Educación  
             14002000029500005    Primaria_incom  
...                                          ...  
141210001789 14121000178900002  Secundaria_incom  
             14121000178900003      Primaria_com  
             14121000178900004    Secundaria_com  
             14121000178900005     Sin Educación  
             14121000178900006      Primaria_com  

[326 rows x 4 columns]

In [ ]:
pd.crosstab(df_per.CONACT_CAT, df_per.INGTRMEN.isna())

INGTRMEN,False,True
CONACT_CAT,,
Trabaja,314413,635
Buscó trabajo,0,7017
No trabaja,0,285998
No especificado,0,2716
Blanco por pase,0,175575


In [ ]:
idx_bpp = df_viv.query("INGTRHOG_CAT == 'Blanco por pase'").index
df_per.loc[idx_bpp][["EDAD", "CONACT_CAT"]].CONACT_CAT.value_counts()

CONACT_CAT
No trabaja         67078
Blanco por pase    13886
Buscó trabajo       2513
No especificado     1267
Trabaja                0
Name: count, dtype: int64

In [ ]:
idx_no_bpp = df_viv.query("INGTRHOG_CAT != 'Blanco por pase'").index
df_per.loc[idx_no_bpp][["EDAD", "CONACT_CAT"]].groupby("ID_VIV").filter(
    lambda df: not np.any(df.CONACT_CAT.isin(["Trabaja"]))
)

,,EDAD,CONACT_CAT
ID_VIV,ID_PERSONA,,


In [ ]:
idx_nan = df_viv.query("INGTRHOG_CAT == 'No especificado'").index
df_per.loc[idx_nan][["EDAD", "CONACT_CAT", "INGTRMEN"]]

EDAD  CONACT_CAT  INGTRMEN
ID_VIV       ID_PERSONA                                   
140020001384 14002000138400001    30     Trabaja      3440
             14002000138400002    77     Trabaja      <NA>
             14002000138400003    59     Trabaja      <NA>
140020001461 14002000146100001    28     Trabaja      4300
             14002000146100002    55  No trabaja      <NA>
...                              ...         ...       ...
141240000653 14124000065300004    60     Trabaja      <NA>
141240001880 14124000188000001    70  No trabaja      <NA>
             14124000188000002    72     Trabaja      <NA>
141240001937 14124000193700001    21     Trabaja      7310
             14124000193700002    48     Trabaja      <NA>

[1902 rows x 3 columns]

In [ ]:
df_per.query("CONACT_CAT == 'Trabaja'").INGTRMEN.isna().sum()

np.int64(635)

In [ ]:
df_per.CONACT_CAT.value_counts()

CONACT_CAT
Trabaja            315048
No trabaja         285998
Blanco por pase    175575
Buscó trabajo        7017
No especificado      2716
Name: count, dtype: int64

In [ ]:
# Para discapacidad, NO especificado-> No tiene, Se desconoce -> NaN

In [ ]:
for col in num_cols_viv:
    df_viv[col] = df_viv[col].astype("Int64")
for col in num_cols_per:
    df_per[col] = df_per[col].astype("Int64")

for col in cat_cols_viv:
    df_viv[col] = df_viv[col].astype("category")
for col in cat_cols_per:
    df_per[col] = df_per[col].astype("category")

# Remove unused categories (e.g. "No especificado" replaced by NaN above)
for col in df_viv.select_dtypes("category").columns:
    df_viv[col] = df_viv[col].cat.remove_unused_categories()
for col in df_per.select_dtypes("category").columns:
    df_per[col] = df_per[col].cat.remove_unused_categories()

# Remove one hot encoded columns and deterministically derived columns
df_viv = df_viv.drop(columns=[c for c in df_viv.columns if "FINANCIAMIENTO_" in c])
df_per = df_per.drop(columns=[c for c in df_per.columns if "MED_TRASLADO_ESC_" in c])
df_per = df_per.drop(columns=[c for c in df_per.columns if "MED_TRASLADO_TRAB_" in c])
df_per = df_per.drop(columns=[c for c in df_per.columns if c.startswith("DHSERSAL_")])
df_per = df_per.drop(columns=["DIS_CON", "DIS_LIMI", "EDUC"])

print(f"Loaded {len(df_viv):,} households and {len(df_per):,} persons.")


We want to impute the columns used for synthesis, we need the constraints to find them.

In [ ]:
from mxcensus import constraints_personas, constraints_viviendas

constraints_viv = constraints_viviendas()
constraints_per = constraints_personas()

We obtain the columns to impute.

In [ ]:
per_cols = set()
for v in constraints_per.values():
    per_cols = per_cols.union(v)

viv_cols = set()
for v in constraints_viv.values():
    viv_cols = viv_cols.union(v)

## Step 1: Variable Inventory and Classification

The two tables share `ID_VIV` as the join key:
- `df_viv`: indexed by `ID_VIV` (one row per household)
- `df_per`: indexed by `(ID_VIV, ID_PERSONA)` (one row per person)

Survey weight for both tables is `FACTOR` (expansion factor, household-level).

Synthesis targets come from the constraints files. Not every synthesis column necessarily has missing values — the missingness check below reveals which ones actually need imputation.

In [ ]:
from typing import Dict, List

# ── Join key and weight columns ───────────────────────────────────────────────
ID_VIV_COL = "ID_VIV"  # index name in df_viv; level 0 of MultiIndex in df_per
HH_WEIGHT_COL = "FACTOR"  # expansion factor (household-level, present in both tables)
PER_WEIGHT_COL = "FACTOR"

# ── High-cardinality design/geography columns (predictor-only, never imputed) ─
design_cols = ["MUN", "LOC50K", "UPM", "ESTRATO", "COBERTURA", "TAMLOC"]

# ── Household imputation targets: all synthesis columns from constraints ───────
hh_impute_targets: List[str] = sorted(viv_cols)

# ── Person imputation targets: constraints reference DHSERSAL_* dummies,
#    but those are dropped from df_per during preprocessing. Replace with
#    source columns DHSERSAL1/DHSERSAL2; re-derive dummies post-loop.
_dhsersal_derived = {c for c in per_cols if c.startswith("DHSERSAL_")}
person_impute_targets: List[str] = sorted(
    (per_cols - _dhsersal_derived) | {"DHSERSAL1", "DHSERSAL2"}
)

# ── High-priority targets: appear in many constraints & have strong
#    within-household correlation → get per-variable LOO refresh ─────────────
person_high_priority_targets: List[str] = ["EDUC", "CONACT_CAT", "SITUA_CONYUGAL_CAT"]

print(f"Household imputation targets  : {len(hh_impute_targets)}")
print(f"Person  imputation targets    : {len(person_impute_targets)}")
print(f"Household targets : {hh_impute_targets}")
print(f"Person targets    : {person_impute_targets}")
print(f"DHSERSAL derived cols removed : {sorted(_dhsersal_derived)}")
print("DHSERSAL source cols added    : ['DHSERSAL1', 'DHSERSAL2']")

Household imputation targets  : 28
Person  imputation targets    : 24
Household targets : ['ABA_AGUA_ENTU', 'AGUA_ENTUBADA', 'AUTOPROP', 'BICICLETA', 'CELULAR', 'CISTERNA', 'CLAVIVP_CAT', 'COMPUTADORA', 'CONAGUA', 'CON_VJUEGOS', 'CUADORM_CAT', 'DRENAJE_CAT', 'ELECTRICIDAD', 'HORNO', 'INTERNET', 'JEFE_SEXO', 'LAVADORA', 'MOTOCICLETA', 'PISOS', 'RADIO', 'REFRIGERADOR', 'SERSAN', 'SERV_PEL_PAGA', 'SERV_TV_PAGA', 'TELEFONO', 'TELEVISOR', 'TINACO', 'TOTCUART_CAT']
Person targets    : ['AFRODES', 'ALFABET', 'ASISTEN', 'CONACT_CAT', 'DHSERSAL1', 'DHSERSAL2', 'DIS_BANARSE', 'DIS_CAMINAR', 'DIS_CON', 'DIS_HABLAR', 'DIS_LIMI', 'DIS_MENTAL', 'DIS_OIR', 'DIS_RECORDAR', 'DIS_VER', 'EDAD_CAT', 'EDUC', 'ENT_PAIS_NAC_CAT', 'ENT_PAIS_RES_CAT', 'HESPANOL', 'HLENGUA', 'RELIGION_CAT', 'SEXO', 'SITUA_CONYUGAL_CAT']
DHSERSAL derived cols removed : ['DHSERSAL_AFIL', 'DHSERSAL_IMSS', 'DHSERSAL_IMSS_Prospera/Bienestar', 'DHSERSAL_ISSSTE', 'DHSERSAL_ISSSTE_E', 'DHSERSAL_No afiliado', 'DHSERSAL_Otro', 'DHSERSAL_

### DHSERSAL re-derivation

`DHSERSAL1` and `DHSERSAL2` are the raw survey answers (one or two health services per person). The `DHSERSAL_*` dummy columns (`DHSERSAL_IMSS`, `DHSERSAL_AFIL`, …) are deterministically derived from them via `dhsersal_create_dummies`.

**Imputation strategy**:
1. Impute `DHSERSAL1` and `DHSERSAL2` directly inside the MICE loop.
2. Call `rederive_dhsersal_dummies()` once after the loop completes to rebuild all `DHSERSAL_*` columns.

`DHSERSAL2` has `"Blanco por pase"` for persons with only one health service — `build_skip_masks` will detect and exclude those rows from imputation automatically.

In [ ]:
from census_mx import rederive_dhsersal_dummies

In [ ]:
from linkedmice.reporting import missing_report_both

missing_report_both(df_viv, df_per, hh_impute_targets, person_impute_targets)


=== Household targets: 26/27 columns have NaN ===
               n_missing  pct_missing
ABA_AGUA_ENTU        116        0.054
BICICLETA             84        0.039
CON_VJUEGOS           82        0.038
MOTOCICLETA           80        0.037
RADIO                 64        0.030
TELEFONO              62        0.029
AUTOPROP              61        0.028
COMPUTADORA           60        0.028
HORNO                 58        0.027
INTERNET              53        0.024
TELEVISOR             50        0.023
SERV_TV_PAGA          49        0.023
LAVADORA              46        0.021
CELULAR               45        0.021
SERV_PEL_PAGA         45        0.021
CISTERNA              43        0.020
REFRIGERADOR          40        0.018
TINACO                33        0.015
PISOS                 31        0.014
SERSAN                23        0.011
TOTCUART_CAT          21        0.010
DRENAJE_CAT           18        0.008
CUADORM_CAT           17        0.008
AGUA_ENTUBADA         15        0.007

## Step 2: Skip Pattern Analysis

In this census, structural non-response is coded as `"Blanco por pase"` (BPP) — distinct from item non-response (`NaN`). BPP rows must **not** be imputed; they are excluded from both training and prediction for the respective column.

`build_skip_masks` detects BPP generically. The cross-validation cell below verifies each condition holds in both directions (BPP → condition and condition → BPP).

**Person-level skip conditions:**

| Column | Skip condition |
|--------|---------------|
| `HLENGUA` | Age < 3 (`EDAD_CAT == "0-2"`) |
| `EDUC` | Age < 3 |
| `ASISTEN` | Age < 3 |
| `ALFABET` | Age < 5 |
| `ENT_PAIS_RES_CAT` | Age < 5 (migration question) |
| `CONACT_CAT` | Age < 12 |
| `SITUA_CONYUGAL_CAT` | Age < 12 |
| `HESPANOL` | `HLENGUA` = `"No"` (monolingual Spanish speakers not asked) |
| `DHSERSAL2` | Already has one service or none (`DHSERSAL1` ≠ multi-service code) |

**Household-level skip conditions:**

| Column | Skip condition |
|--------|---------------|
| `ABA_AGUA_ENTU` | `AGUA_ENTUBADA == "No tiene"` (no piped water → source question skipped) |
| `CONAGUA` | `SERSAN == "No tienen taza de baño ni letrina."` (no sanitary facility → flush question skipped) |
| Most other household targets | `CLAVIVP_CAT == "Otro"` (non-residential dwelling, 491 rows)

In [ ]:
from typing import Callable, List, Tuple

# Type alias: (target_col, condition_fn, human_label)
# condition_fn(df) → bool Series — True where the skip IS expected (BPP expected).
SkipCondition = Tuple[str, Callable, str]

# ── Person-level skip conditions ──────────────────────────────────────────────
# Age-band constants (inlined; EDAD_CAT categories after preprocessing).
_AGE_0_4 = ["0-2", "3-4"]
_AGE_0_7 = ["0-2", "3-4", "5", "6-7"]
_AGE_0_11 = ["0-2", "3-4", "5", "6-7", "8-11"]

SKIP_CONDITIONS_PER: List[SkipCondition] = [
    ("HLENGUA", lambda df: df["EDAD_CAT"] == "0-2", "EDAD_CAT < 3"),
    ("EDUC", lambda df: df["EDAD_CAT"] == "0-2", "EDAD_CAT < 3"),
    ("ASISTEN", lambda df: df["EDAD_CAT"] == "0-2", "EDAD_CAT < 3"),
    ("ALFABET", lambda df: df["EDAD_CAT"].isin(_AGE_0_4), "EDAD_CAT < 5"),
    ("ENT_PAIS_RES_CAT", lambda df: df["EDAD_CAT"].isin(_AGE_0_4), "EDAD_CAT < 5"),
    ("CONACT_CAT", lambda df: df["EDAD_CAT"].isin(_AGE_0_11), "EDAD_CAT < 12"),
    ("SITUA_CONYUGAL_CAT", lambda df: df["EDAD_CAT"].isin(_AGE_0_11), "EDAD_CAT < 12"),
    (
        "HESPANOL",
        lambda df: (df["HLENGUA"] == "No") | (df["EDAD_CAT"] == "0-2"),
        "HLENGUA != 'Sí' | EDAD_CAT < 3",
    ),
    # DHSERSAL2: skip when person has only one health-service enrollment.
    # Condition is questionnaire-routing logic (DHSERSAL1 answer count),
    # not a direct column comparison — omitted from automated validation.
]

# ── Household-level skip conditions ──────────────────────────────────────────
# Base skip: non-residential dwelling → most HH questions inapplicable.
# Excluded from the base group:
#   CLAVIVP_CAT  — the parent column itself (0 BPP rows)
#   JEFE_SEXO    — head attributes recorded even for non-residential units
#   ABA_AGUA_ENTU / CONAGUA — compound conditions defined separately below
_BASE_SKIP_HH_COLS = [
    c
    for c in hh_impute_targets
    if c not in {"CLAVIVP_CAT", "JEFE_SEXO", "ABA_AGUA_ENTU", "CONAGUA"}
]

SKIP_CONDITIONS_HH: List[SkipCondition] = [
    # Base skip: one entry per affected column
    *[
        (col, lambda df: df["CLAVIVP_CAT"] == "Otro", "CLAVIVP_CAT == 'Otro'")
        for col in _BASE_SKIP_HH_COLS
    ],
    # ABA_AGUA_ENTU: skip when no piped water OR non-residential dwelling
    (
        "ABA_AGUA_ENTU",
        lambda df: (df["CLAVIVP_CAT"] == "Otro") | (df["AGUA_ENTUBADA"] == "No tiene"),
        "CLAVIVP_CAT=='Otro' | AGUA_ENTUBADA=='No tiene'",
    ),
    # CONAGUA: skip when no sanitary facility OR non-residential dwelling
    (
        "CONAGUA",
        lambda df: (
            (df["CLAVIVP_CAT"] == "Otro")
            | (df["SERSAN"] == "No tienen taza de baño ni letrina.")
        ),
        "CLAVIVP_CAT=='Otro' | SERSAN=='No tienen taza...'",
    ),
]

print(f"SKIP_CONDITIONS_PER : {len(SKIP_CONDITIONS_PER)} conditions")
print(
    f"SKIP_CONDITIONS_HH  : {len(SKIP_CONDITIONS_HH)} conditions  "
    f"({len(_BASE_SKIP_HH_COLS)} base-skip + 2 compound)"
)


SKIP_CONDITIONS_PER : 8 conditions
SKIP_CONDITIONS_HH  : 26 conditions  (24 base-skip + 2 compound)


In [ ]:
from linkedmice.utils import repair_parent_child_nan

# Parent-NaN → child-NaN repair
# When a parent question is NaN (item non-response), the skip condition for its
# child cannot be determined, so the child must also be NaN (not BPP).
parent_child_pairs_hh = [
    ("AGUA_ENTUBADA", "ABA_AGUA_ENTU"),
    ("SERSAN", "CONAGUA"),
]
df_viv = repair_parent_child_nan(df_viv, parent_child_pairs_hh)

parent_child_pairs_per = [
    ("HLENGUA", "HESPANOL"),
]
df_per = repair_parent_child_nan(df_per, parent_child_pairs_per)


Repaired ABA_AGUA_ENTU: 15 rows reset from BPP → NaN (parent AGUA_ENTUBADA is NaN)
Repaired CONAGUA: 23 rows reset from BPP → NaN (parent SERSAN is NaN)
Repaired HESPANOL: 299 rows reset from BPP → NaN (parent HLENGUA is NaN)


In [ ]:
from linkedmice.reporting import bpp_report

BPP = "Blanco por pase"

# ── Audit: which imputation targets contain BPP? ─────────────────────────────
bpp_report(df_viv, df_per, hh_impute_targets, person_impute_targets, BPP)

Person targets with 'Blanco por pase' rows:
  ALFABET                                          70,601  ( 8.98%)
  ASISTEN                                          40,924  ( 5.20%)
  CONACT_CAT                                      175,575  (22.33%)
  DHSERSAL2                                       776,725  (98.78%)
  EDUC                                             40,924  ( 5.20%)
  ENT_PAIS_RES_CAT                                 70,601  ( 8.98%)
  HESPANOL                                        762,750  (97.00%)
  HLENGUA                                          40,924  ( 5.20%)
  SITUA_CONYUGAL_CAT                              175,575  (22.33%)

Household targets with 'Blanco por pase' rows:
  ABA_AGUA_ENTU                                     4,820  ( 2.23%)
  AGUA_ENTUBADA                                       491  ( 0.23%)
  AUTOPROP                                            491  ( 0.23%)
  BICICLETA                                           491  ( 0.23%)
  CELULAR               

In [ ]:
from linkedmice.mice import build_skip_masks, build_missing_masks

person_skip_masks = build_skip_masks(df_per, person_impute_targets)
hh_skip_masks = build_skip_masks(df_viv, hh_impute_targets)

# Build the final missing masks (NaN AND NOT a structural skip)
# Confirm they are always disjoint — a cell can't be both BPP and NaN
person_missing_masks = build_missing_masks(
    df_per, person_impute_targets, person_skip_masks
)
hh_missing_masks = build_missing_masks(df_viv, hh_impute_targets, hh_skip_masks)


Skip mask summary (% structural skips per column):
  ALFABET                                          70,601  (  9.0%)
  ASISTEN                                          40,924  (  5.2%)
  CONACT_CAT                                      175,575  ( 22.3%)
  DHSERSAL2                                       776,725  ( 98.8%)
  EDUC                                             40,924  (  5.2%)
  ENT_PAIS_RES_CAT                                 70,601  (  9.0%)
  HESPANOL                                        762,750  ( 97.0%)
  HLENGUA                                          40,924  (  5.2%)
  SITUA_CONYUGAL_CAT                              175,575  ( 22.3%)

Skip mask summary (% structural skips per column):
  ABA_AGUA_ENTU                                     4,820  (  2.2%)
  AGUA_ENTUBADA                                       491  (  0.2%)
  AUTOPROP                                            491  (  0.2%)
  BICICLETA                                           491  (  0.2%)
  CELULAR   

In [ ]:
from linkedmice.reporting import cross_validate_all_skips

assert cross_validate_all_skips(SKIP_CONDITIONS_PER, df_per, "Person table")
assert cross_validate_all_skips(SKIP_CONDITIONS_HH, df_viv, "Household table")



=== Person table — 8 skip condition(s) ===
  column                                  n_bpp   bpp\cond   cond\bpp  fwd bwd
  --------------------------------------------------------------------------
  HLENGUA                                40,924          0          0   ✓   ✓
  EDUC                                   40,924          0          0   ✓   ✓
  ASISTEN                                40,924          0          0   ✓   ✓
  ALFABET                                70,601          0          0   ✓   ✓
  ENT_PAIS_RES_CAT                       70,601          0          0   ✓   ✓
  CONACT_CAT                            175,575          0          0   ✓   ✓
  SITUA_CONYUGAL_CAT                    175,575          0          0   ✓   ✓
  HESPANOL                              762,750          0          0   ✓   ✓

  All 8 condition(s) passed.

=== Household table — 26 skip condition(s) ===
  column                                  n_bpp   bpp\cond   cond\bpp  fwd bwd
  -----------------

### Dependent skip refresh

Some skip conditions depend on a variable that is itself imputed. If the parent variable changes from NaN to a value that triggers (or releases) a skip, the child's skip mask and stored value must be updated immediately — before the child is imputed in the same iteration.

**Dependency pairs in this dataset:**

| Parent | Skip triggers when… | Child | Direction |
|--------|---------------------|-------|-----------|
| `AGUA_ENTUBADA` | `== "No tiene"` | `ABA_AGUA_ENTU` | HH → HH |
| `SERSAN` | `== "No tienen taza de baño ni letrina."` | `CONAGUA` | HH → HH |
| `HLENGUA` | `!= "Sí"` | `HESPANOL` | Per → Per |

`refresh_dependent_skips` is called in the MICE loop immediately after imputing the parent column. It handles two cases:

- **Parent → skip value**: child is forced to BPP and removed from `missing_mask`.
- **Parent → non-skip value**: if child was BPP (left over from the initial state), it is set to NaN and added back to `missing_mask` so it gets imputed in this iteration.

An end-of-iteration `apply_consistency_repair` call sweeps up any residual mismatches.

In [ ]:
from linkedmice.mice import refresh_dependent_skips, apply_consistency_repair

# ── Skip dependency definitions ───────────────────────────────────────────────
# Each entry: (parent_col, skip_predicate, child_col)
# skip_predicate(series) → bool mask: True where the skip IS triggered.

HH_SKIP_DEPS: List[tuple] = [
    ("AGUA_ENTUBADA", lambda s: s == "No tiene", "ABA_AGUA_ENTU"),
    ("SERSAN", lambda s: s == "No tienen taza de baño ni letrina.", "CONAGUA"),
]

PER_SKIP_DEPS: List[tuple] = [
    ("HLENGUA", lambda s: s == "No", "HESPANOL"),
]

print("refresh_dependent_skips and apply_consistency_repair defined.")
print(f"HH  skip dependencies : {[(p, c) for p, _, c in HH_SKIP_DEPS]}")
print(f"Per skip dependencies : {[(p, c) for p, _, c in PER_SKIP_DEPS]}")

refresh_dependent_skips and apply_consistency_repair defined.
HH  skip dependencies : [('AGUA_ENTUBADA', 'ABA_AGUA_ENTU'), ('SERSAN', 'CONAGUA')]
Per skip dependencies : [('HLENGUA', 'HESPANOL')]


## Step 3: Initialization

Before iteration 1, fill each NaN cell with a random draw from the **observed** values of that column (excluding both `NaN` and `"Blanco por pase"`).  This gives the chained-equations regressions a non-trivial starting state.

Working copies `hh_work` / `per_work` receive the fills; originals `df_viv` / `df_per` stay pristine.  The missing-mask dicts (`hh_missing_mask`, `person_missing_mask`) captured in Step 2 remain the authoritative source of truth for which cells to re-impute in every iteration.

In [ ]:
from linkedmice.mice import initial_fill
from linkedmice.reporting import initial_fill_report

# ── Create working copies (originals stay pristine) ───────────────────────────
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

hh_work = initial_fill(df_viv, hh_impute_targets, hh_missing_masks, hh_skip_masks, rng)
per_work = initial_fill(
    df_per, person_impute_targets, person_missing_masks, person_skip_masks, rng
)

initial_fill_report(
    hh_work,
    per_work,
    hh_missing_masks,
    person_missing_masks,
    hh_skip_masks,
    person_skip_masks,
    hh_impute_targets,
    person_impute_targets,
)


Post-fill NaN check:
  All target NaN cells filled.

BPP preservation check:
  All BPP rows preserved.

Total cells filled — HH: 1,251  |  Person: 12,942


## Step 4: Cross-Table Feature Engineering

Features are computed from the current state of both tables and refreshed at defined points during the MICE loop (see design decision D2 in the plan).

### 4a–e — Functions implemented here
| Function | Direction | Refresh timing |
|---|---|---|
| `compute_person_aggregates` | Person → Household | Start of each HH block |
| `broadcast_household_attrs` | Household → Person | Start of each Person block |
| `compute_role_features` | Person → Person | Start of each Person block |
| `leave_one_out_category_counts` | Person → Person | Per-variable for high-priority targets |
| `compute_position_features` | Person → Person | Start of each Person block |

`PARENTESCO` category values (post-preprocessing) used to identify roles:
- **Head**: `"Jefa(e)"`
- **Partner**: `"Esposa(o)"`, `"Concubina(o) o unión libre"`, `"Amante o querida(o)"`
- **Child**: `"Hija(o)"`, `"Hija(o) adoptiva(o)"`, `"Hijastra(o)"`, `"Hija(o) de crianza"`

In [ ]:
from census_mx import compute_person_aggregates

# Quick smoke-test
_aggs_test = compute_person_aggregates(df_per)
print("compute_person_aggregates — shape:", _aggs_test.shape)
print(_aggs_test.head(3))

compute_person_aggregates — shape: (216458, 12)
              n_children_per  n_adults_per  n_hombres_per  n_trabaja_per  \
ID_VIV                                                                     
140010000001               1             2              1              2   
140010000002               3             1              1              1   
140010000003               1             4              3              4   

              mean_escoacum  has_trabaja_per head_sexo_per head_edad_cat_per  \
ID_VIV                                                                         
140010000001           12.0                1        Hombre             25-49   
140010000002           7.75                1         Mujer             25-49   
140010000003           11.4                1        Hombre             50-59   

                 head_educ_per head_conact_per  \
ID_VIV                                           
140010000001  Secundaria_incom         Trabaja   
140010000002    Secu

In [ ]:
from linkedmice.feature_eng import broadcast_household_attrs

# ── Household columns to broadcast into the person table ─────────────────────
HH_BROADCAST_COLS: List[str] = [
    "INGTRHOG_CAT",  # household income bracket
    "TIPOHOG",  # household type (nuclear, extended, …)
    "CLAVIVP_CAT",  # dwelling class
    "NUMPERS",  # number of persons in dwelling (census count),
    "COMBUSTIBLE",
    "ALIMENTACION",
    "MCONMIG",
    "TENENCIA",
    "SERSAN",
    "DRENAJE_CAT",
]

# ── Smoke-test ───────────────────────────────────────────────────────────────
_per_enriched = broadcast_household_attrs(df_per, df_viv, HH_BROADCAST_COLS)
new_hh_cols = [c for c in _per_enriched.columns if c.endswith("_hh")]
print("Broadcast columns added:", new_hh_cols)
print(_per_enriched[new_hh_cols].head(3))

Broadcast columns added: ['INGTRHOG_CAT_hh', 'TIPOHOG_hh', 'CLAVIVP_CAT_hh', 'NUMPERS_hh', 'COMBUSTIBLE_hh', 'ALIMENTACION_hh', 'MCONMIG_hh', 'TENENCIA_hh', 'SERSAN_hh', 'DRENAJE_CAT_hh']
                               INGTRHOG_CAT_hh                TIPOHOG_hh  \
ID_VIV       ID_PERSONA                                                    
140010000001 14001000000100001   10,000-19,999  Hogar Nuclear (Familiar)   
             14001000000100002   10,000-19,999  Hogar Nuclear (Familiar)   
             14001000000100003   10,000-19,999  Hogar Nuclear (Familiar)   

                               CLAVIVP_CAT_hh  NUMPERS_hh   COMBUSTIBLE_hh  \
ID_VIV       ID_PERSONA                                                      
140010000001 14001000000100001           Otro           3  Blanco por pase   
             14001000000100002           Otro           3  Blanco por pase   
             14001000000100003           Otro           3  Blanco por pase   

                               ALIMENTAC

In [ ]:
from census_mx import compute_role_features, HEAD_VALS

# ── Smoke-test ────────────────────────────────────────────────────────────────
_role_test = compute_role_features(df_per)
print("compute_role_features — shape:", _role_test.shape)
print(_role_test.head(5))

# Verify heads have NaN in their own head_* columns
_is_head_test = df_per["PARENTESCO"].isin(HEAD_VALS)
assert _role_test.loc[_is_head_test, "head_educ"].isna().all(), (
    "head_educ should be NaN for heads"
)
print("Self-leakage check passed.")

compute_role_features — shape: (786354, 12)
                                       head_educ head_sexo head_edad_cat  \
ID_VIV       ID_PERSONA                                                    
140010000001 14001000000100001  Secundaria_incom    Hombre         25-49   
             14001000000100002               NaN       NaN           NaN   
             14001000000100003  Secundaria_incom    Hombre         25-49   
140010000002 14001000000200001               NaN       NaN           NaN   
             14001000000200002    Secundaria_com     Mujer         25-49   

                               head_conact_cat  \
ID_VIV       ID_PERSONA                          
140010000001 14001000000100001         Trabaja   
             14001000000100002             NaN   
             14001000000100003         Trabaja   
140010000002 14001000000200001             NaN   
             14001000000200002      No trabaja   

                                                                       h

In [ ]:
from linkedmice.feature_eng import leave_one_out_category_counts

# ── Smoke-tests ───────────────────────────────────────────────────────────────
_sample = df_per

_loo = leave_one_out_category_counts(_sample, "EDUC")
print("LOO EDUC — shape:", _loo.shape, "| columns:", _loo.columns.tolist())
# Sanity: single-person households should have 0 for all LOO columns
_single_hh = _sample[_sample.groupby(level="ID_VIV").transform("size") == 1]
assert (_loo.loc[_single_hh.index] == 0).all().all(), "Single-person HH must have LOO=0"
print("LOO single-HH check passed.")


LOO EDUC — shape: (786354, 7) | columns: ['n_other_EDUC_Sin Educación', 'n_other_EDUC_Primaria_incom', 'n_other_EDUC_Primaria_com', 'n_other_EDUC_Secundaria_incom', 'n_other_EDUC_Secundaria_com', 'n_other_EDUC_Posbásica', 'n_other_EDUC_Blanco por pase']
LOO single-HH check passed.


In [ ]:
from census_mx import compute_position_features

_pos = compute_position_features(_sample)
print("\nPosition features — shape:", _pos.shape)
print(_pos.head(3))


Position features — shape: (786354, 5)
                                is_head_pos  is_partner_pos  is_child_pos  \
ID_VIV       ID_PERSONA                                                     
140010000001 14001000000100001            0               1             0   
             14001000000100002            1               0             0   
             14001000000100003            0               0             1   

                                hh_size_pos  is_single_person_hh  
ID_VIV       ID_PERSONA                                           
140010000001 14001000000100001            3                    0  
             14001000000100002            3                    0  
             14001000000100003            3                    0  


## Step 5: Backend-Agnostic Weighted Imputation Function

`weighted_impute_categorical` is the per-column primitive of the MICE loop:
1. **Training set**: observed rows (`~missing_mask & ~skip_mask`), weighted by `FACTOR`.
2. **Prediction set**: NaN rows (`missing_mask` — BPP rows are never in this mask).
3. **Stochastic sampling**: draw a category per row from the predicted class probabilities (distribution-preserving, not argmax).

Three backends (LightGBM primary, CatBoost/XGBoost optional for Step 11 bake-off).

In [ ]:
from linkedmice.mice import weighted_impute_categorical

In [ ]:
# ── Step 5 smoke test on synthetic data ──────────────────────────────────────
_rng_st = np.random.default_rng(7)
_n_st = 800

_st_df = pd.DataFrame(
    {
        "x_cat": pd.Categorical(
            _rng_st.choice(["A", "B", "C"], _n_st), categories=["A", "B", "C"]
        ),
        "x_num": pd.Series(_rng_st.integers(0, 10, _n_st)).astype(
            "Int64"
        ),  # nullable int
        "y": pd.Categorical(
            _rng_st.choice(["Yes", "No", BPP], _n_st, p=[0.45, 0.45, 0.10]),
            categories=["Yes", "No", BPP],
        ),
        "w": np.ones(_n_st),
    }
)

# Inject 60 NaN (item non-response) and keep existing BPP rows
_bpp_mask_st = _st_df["y"] == BPP
_nan_cands = _st_df.index[~_bpp_mask_st]
_nan_idx_st = _rng_st.choice(_nan_cands, size=60, replace=False)
_st_df.loc[_nan_idx_st, "y"] = pd.NA
_miss_st = _st_df["y"].isna()
_skip_st = _st_df["y"] == BPP

# BPP ∩ NaN must be empty
assert (_miss_st & _skip_st).sum() == 0

_imp_st = weighted_impute_categorical(
    df=_st_df,
    target_col="y",
    predictor_cols=["x_cat", "x_num"],
    missing_mask=_miss_st,
    skip_mask=_skip_st,
    weights=_st_df["w"],
    rng=_rng_st,
    backend="lightgbm",
)

assert len(_imp_st) == 60, f"Expected 60 imputed rows, got {len(_imp_st)}"
assert set(_imp_st.values).issubset({"Yes", "No"}), (
    "BPP must not appear in imputed values"
)
# BPP rows must remain untouched
_st_df.loc[_miss_st, "y"] = _imp_st.values
assert (_st_df.loc[_skip_st, "y"] == BPP).all(), "BPP rows modified"

print(f"Smoke test passed — 60 rows imputed: {dict(_imp_st.value_counts())}")

Smoke test passed — 60 rows imputed: {'Yes': np.int64(34), 'No': np.int64(26), 'Blanco por pase': np.int64(0)}


## Step 6: Predictor Registry

For each impute target, the predictor set is all columns currently in the working DataFrame **except**:
- The target itself.
- `FACTOR` (the weight — used as sample weight, not a predictor).
- `DHSERSAL_*` derived dummy columns in the person table — they are stale during the loop (not updated until post-loop re-derivation) and are derived from two targets (`DHSERSAL1`/`DHSERSAL2`), creating leakage.

The registry is rebuilt on each iteration call after feature columns are refreshed, so newly added engineered features are automatically included.

In [ ]:
from linkedmice.mice import build_predictor_registry

# Fixed exclusion sets
_HH_EXCLUDE: set = {HH_WEIGHT_COL}  # {'FACTOR'}
_PER_EXCLUDE: set = {
    PER_WEIGHT_COL
} | _dhsersal_derived  # {'FACTOR'} + stale DHSERSAL_* dummies

# ── Smoke test: verify exclusions are respected ───────────────────────────────
_reg_hh = build_predictor_registry(hh_work, hh_impute_targets, _HH_EXCLUDE)
_reg_per = build_predictor_registry(per_work, person_impute_targets, _PER_EXCLUDE)

# FACTOR must not appear in any predictor list
assert all(HH_WEIGHT_COL not in v for v in _reg_hh.values()), (
    "FACTOR in HH predictor set"
)
assert all(PER_WEIGHT_COL not in v for v in _reg_per.values()), (
    "FACTOR in Per predictor set"
)
# No DHSERSAL_* derived columns in person predictor sets
for target, preds in _reg_per.items():
    bad = [p for p in preds if p.startswith("DHSERSAL_")]
    assert not bad, f"Stale DHSERSAL_* in predictor set for {target}: {bad}"
# Target itself must not appear in its own predictor list
for target, preds in {**_reg_hh, **_reg_per}.items():
    assert target not in preds, f"Target {target} appears in its own predictor list"

print(
    f"HH  registry: {len(_reg_hh)} targets, avg {sum(len(v) for v in _reg_hh.values()) / len(_reg_hh):.0f} predictors each"
)
print(
    f"Per registry: {len(_reg_per)} targets, avg {sum(len(v) for v in _reg_per.values()) / len(_reg_per):.0f} predictors each"
)
print("Predictor registry checks passed.")

HH  registry: 27 targets, avg 80 predictors each
Per registry: 24 targets, avg 85 predictors each
Predictor registry checks passed.


## Step 7: Integrated MICE Loop

Each iteration:
1. **HH block** — refresh person aggregates → impute HH targets (ascending missingness) → update dependent skips.
2. **Person block** — refresh HH broadcasts, role features, position features, LOO features → impute person targets (ascending missingness) → update dependent skips; re-refresh LOO for high-priority targets just before they are imputed.
3. **End-of-iteration** — `apply_consistency_repair` for both tables.
4. **Diagnostics** — record per-variable distributions for the originally-missing cells.

`integrated_mice` always works on deep copies of the input DataFrames and masks — the caller's state is unchanged.

In [ ]:
from linkedmice.mice import integrated_mice

### Pilot run — 10 % subsample, 2 iterations

Verifies end-to-end correctness and gives a per-iteration runtime estimate before the full run.  Stratified by whole households so person–household linkage is preserved.

In [ ]:
# ── Build 10% subsample ───────────────────────────────────────────────────────
_rng_pilot = np.random.default_rng(99)
_all_hh_ids = hh_work.index.values
_n_pilot = max(1, int(len(_all_hh_ids) * 0.10))
_pilot_hh_ids = set(_rng_pilot.choice(_all_hh_ids, size=_n_pilot, replace=False))

hh_pilot = hh_work.loc[sorted(_pilot_hh_ids)].copy()

_pilot_per_mask = per_work.index.get_level_values("ID_VIV").isin(_pilot_hh_ids)
per_pilot = per_work.loc[_pilot_per_mask].copy()

hh_mm_pilot = {
    c: hh_missing_masks[c].loc[hh_pilot.index]
    for c in hh_impute_targets
    if c in hh_work.columns
}
per_mm_pilot = {
    c: person_missing_masks[c].loc[per_pilot.index]
    for c in person_impute_targets
    if c in per_work.columns
}
hh_sm_pilot = {
    c: hh_skip_masks[c].loc[hh_pilot.index]
    for c in hh_impute_targets
    if c in hh_work.columns
}
per_sm_pilot = {
    c: person_skip_masks[c].loc[per_pilot.index]
    for c in person_impute_targets
    if c in per_work.columns
}

print(f"Pilot subsample: {len(hh_pilot):,} households  |  {len(per_pilot):,} persons")
print(
    f"Missing cells — HH: {sum(v.sum() for v in hh_mm_pilot.values()):,}  "
    f"Per: {sum(v.sum() for v in per_mm_pilot.values()):,}"
)

# ── Run pilot ─────────────────────────────────────────────────────────────────
(
    hh_pilot_out,
    per_pilot_out,
    hh_mm_pilot_out,
    per_mm_pilot_out,
    hh_sm_pilot_out,
    per_sm_pilot_out,
    pilot_diag,
) = integrated_mice(
    hh_df=hh_pilot,
    per_df=per_pilot,
    hh_missing_mask_in=hh_mm_pilot,
    person_missing_mask_in=per_mm_pilot,
    hh_skip_masks_in=hh_sm_pilot,
    person_skip_masks_in=per_sm_pilot,
    hh_impute_targets=hh_impute_targets,
    person_impute_targets=person_impute_targets,
    hh_exclude=_HH_EXCLUDE,
    per_exclude=_PER_EXCLUDE,
    hh_skip_deps=HH_SKIP_DEPS,
    per_skip_deps=PER_SKIP_DEPS,
    person_high_priority_targets=person_high_priority_targets,
    hh_broadcast_cols=HH_BROADCAST_COLS,
    n_iterations=2,
    backend="lightgbm",
    rng=np.random.default_rng(RANDOM_SEED),
    verbose=True,
)

# ── Post-run sanity checks ────────────────────────────────────────────────────
print("\n─── Post-pilot sanity checks ───")

# 1. No NaN remaining in any originally-missing cell
for col in hh_impute_targets:
    if col not in hh_pilot_out.columns:
        continue
    still_nan = hh_mm_pilot[col] & hh_pilot_out[col].isna()
    assert not still_nan.any(), (
        f"HH {col}: {still_nan.sum()} NaN remaining after imputation"
    )
for col in person_impute_targets:
    if col not in per_pilot_out.columns:
        continue
    still_nan = per_mm_pilot[col] & per_pilot_out[col].isna()
    assert not still_nan.any(), (
        f"Per {col}: {still_nan.sum()} NaN remaining after imputation"
    )
print("  ✓ No residual NaN in imputed cells")

# 2. BPP rows remain BPP — use returned skip masks which include dynamically-added rows
for col in hh_impute_targets:
    if col not in hh_pilot_out.columns or not hasattr(hh_pilot_out[col], "cat"):
        continue
    was_bpp = hh_sm_pilot_out[col]
    assert (hh_pilot_out.loc[was_bpp, col] == BPP).all(), f"HH {col}: BPP rows modified"
for col in person_impute_targets:
    if col not in per_pilot_out.columns or not hasattr(per_pilot_out[col], "cat"):
        continue
    was_bpp = per_sm_pilot_out[col]
    assert (per_pilot_out.loc[was_bpp, col] == BPP).all(), (
        f"Per {col}: BPP rows modified"
    )
print("  ✓ All BPP rows preserved")


Pilot subsample: 21,645 households  |  78,197 persons
Missing cells — HH: 145  Per: 1,528

Iteration 1/2  backend=lightgbm
  HH  AGUA_ENTUBADA  (2 missing)
  HH  CUADORM_CAT  (2 missing)
  HH  DRENAJE_CAT  (2 missing)
  HH  ELECTRICIDAD  (2 missing)
  HH  TOTCUART_CAT  (2 missing)
  HH  CISTERNA  (3 missing)
  HH  PISOS  (3 missing)
  HH  SERSAN  (3 missing)
  HH  CONAGUA  (52 missing)
  HH  INTERNET  (4 missing)
  HH  AUTOPROP  (5 missing)
  HH  RADIO  (5 missing)
  HH  REFRIGERADOR  (5 missing)
  HH  SERV_PEL_PAGA  (5 missing)
  HH  TELEVISOR  (5 missing)
  HH  TINACO  (5 missing)
  HH  CELULAR  (6 missing)
  HH  COMPUTADORA  (6 missing)
  HH  HORNO  (6 missing)
  HH  SERV_TV_PAGA  (6 missing)
  HH  LAVADORA  (7 missing)
  HH  BICICLETA  (10 missing)
  HH  TELEFONO  (10 missing)
  HH  CON_VJUEGOS  (11 missing)
  HH  MOTOCICLETA  (12 missing)
  HH  ABA_AGUA_ENTU  (62 missing)
  Per EDAD_CAT  (2 missing)
  Per SITUA_CONYUGAL_CAT  (25 missing)
  Per ENT_PAIS_RES_CAT  (28 missing)
  Per 

## Step 8: Convergence Diagnostics

`analyze_convergence` computes the total-variation (TV) distance between consecutive iteration snapshots for every imputed variable. Convergence is stable when all per-variable TV distances drop below ~0.01.

In [ ]:
from linkedmice.diagnostics import analyze_convergence

# Apply to pilot diagnostics
pilot_conv = analyze_convergence(pilot_diag, threshold=0.01)


## Step 9: Post-Imputation Consistency Repair

After the loop, run a final consistency pass:
1. Re-apply `apply_consistency_repair` for dynamic skip dependencies (AGUA_ENTUBADA→ABA_AGUA_ENTU, SERSAN→CONAGUA, HLENGUA→HESPANOL).
2. Verify all static BPP rows are still BPP (invariant guaranteed by construction but worth asserting).

For this dataset the loop's end-of-iteration `apply_consistency_repair` already handles all known inconsistencies; this step is a safety net.

In [ ]:
from linkedmice.mice import post_imputation_repair
# post_imputation_repair is now in linkedmice.mice
# Signature: post_imputation_repair(hh_df, per_df, hh_mm, per_mm, hh_sm, per_sm,
#                                   hh_skip_deps, per_skip_deps)


## Step 10: Validation

Compare the weighted distribution of imputed cells against the distribution of observed cells for each imputed variable. A large divergence suggests the imputation model is over- or under-representing certain categories.

TV distance is the primary metric: TV = 0.5 * Σ|obs_prop - imp_prop|.

In [ ]:
from linkedmice.reporting import validate_marginals, run_validation_report
# Both functions are now in linkedmice.reporting
# run_validation_report now takes hh_impute_targets, person_impute_targets,
# hh_weight_col, per_weight_col as explicit parameters.


## Step 11: Backend Bake-Off Infrastructure

Compares imputation backends (LightGBM vs. CatBoost) by artificially masking 5% of observed values per target and measuring weighted accuracy and TV distance on the held-out ground truth.

Run on the 10% pilot subsample to keep runtime manageable.

In [ ]:
from linkedmice.evaluation import (
    create_validation_mask,
    evaluate_imputation,
    print_bakeoff_summary,
)
# See notebooks/imputation_pilot.ipynb for the full bake-off execution.


## Full Run — 5 Iterations on Complete Data

Run the integrated MICE loop on the full `hh_work` / `per_work` (all ~217k households, ~787k persons). Expected runtime: 1–3 hours per iteration depending on hardware.

The cell below is ready to execute. Results land in `hh_imputed` and `per_imputed`.

In [ ]:
(
    hh_imputed,
    per_imputed,
    hh_mm_mice,
    per_mm_mice,
    hh_sm_mice,
    per_sm_mice,
    full_diag,
) = integrated_mice(
    hh_df=hh_work,
    per_df=per_work,
    hh_missing_mask_in=hh_missing_masks,
    person_missing_mask_in=person_missing_masks,
    hh_skip_masks_in=hh_skip_masks,
    person_skip_masks_in=person_skip_masks,
    hh_impute_targets=hh_impute_targets,
    person_impute_targets=person_impute_targets,
    hh_exclude=_HH_EXCLUDE,
    per_exclude=_PER_EXCLUDE,
    hh_skip_deps=HH_SKIP_DEPS,
    per_skip_deps=PER_SKIP_DEPS,
    person_high_priority_targets=person_high_priority_targets,
    hh_broadcast_cols=HH_BROADCAST_COLS,
    n_iterations=5,
    backend="lightgbm",
    rng=np.random.default_rng(RANDOM_SEED),
    verbose=True,
)

full_conv = analyze_convergence(full_diag, threshold=0.01)

hh_imputed, per_imputed, hh_mm_final, per_mm_final, hh_sm_final, per_sm_final = (
    post_imputation_repair(
        hh_imputed,
        per_imputed,
        hh_mm_mice,
        per_mm_mice,
        hh_sm_mice,
        per_sm_mice,
        hh_skip_deps=HH_SKIP_DEPS,
        per_skip_deps=PER_SKIP_DEPS,
    )
)

print(f"\nhh_imputed  shape: {hh_imputed.shape}")
print(f"per_imputed shape: {per_imputed.shape}")


## Step 12: Hand-Off

1. Re-derive `DHSERSAL_*` dummy columns from the imputed `DHSERSAL1`/`DHSERSAL2`.
2. Run the final validation report.
3. Export both tables to Parquet.

In [ ]:
# 1. Re-derive DHSERSAL_* dummies from imputed source columns
per_imputed = rederive_dhsersal_dummies(per_imputed)
new_dhsersal_cols = sorted(c for c in per_imputed.columns if c.startswith("DHSERSAL_"))
print("Re-derived DHSERSAL columns:", new_dhsersal_cols)

# 2. Validation report
hh_tv = run_validation_report(
    hh_imputed,
    per_imputed,
    hh_mm_final,
    per_mm_final,
    hh_sm_final,
    per_sm_final,
    hh_impute_targets,
    person_impute_targets,
)

# 3. Export
from pathlib import Path

_out_dir = Path("../data/census_2020/cuestionario_ampliado/Censo2020_CA_jal_csv")
_out_dir.mkdir(parents=True, exist_ok=True)

hh_imputed.to_parquet(_out_dir / "Viviendas14_imputed.parquet")
per_imputed.to_parquet(_out_dir / "Personas14_imputed.parquet")
print(f"Saved imputed tables to {_out_dir}")
print(f"  Viviendas14_imputed.parquet : {hh_imputed.shape}")
print(f"  Personas14_imputed.parquet  : {per_imputed.shape}")
